In [24]:
import json
from pathlib import Path
from lightgbm import LGBMRegressor
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from catboost import CatBoostRegressor


from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    AdaBoostRegressor,
)

from sklearn.linear_model import (
    Ridge,
    ElasticNet,
    Lasso,
    SGDRegressor,
)

from xgboost import XGBRegressor

from catboost import CatBoostRegressor

from lightgbm import LGBMRegressor

from sklearn.neural_network import MLPRegressor
import unicodedata
import re

In [25]:

from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import Pool

ARQUIVO_PARQUET = "dados/cnpqBolsasAuxilios.parquet"
TARGET = "valor_pago"

print("\nCarregando dataset...")

df = pd.read_parquet(ARQUIVO_PARQUET)

print("Shape original:", df.shape)

# opcional -> amostra para acelerar
df = df.sample(n=5000, random_state=42)
print("Shape após dropna:", df.shape)
q = df['valor_pago'].quantile(0.95)
df = df[df['valor_pago'] <= q]
df = df.drop(columns=['_record_number'])





Carregando dataset...
Shape original: (3274778, 21)
Shape após dropna: (5000, 21)


In [26]:
mapa_codigos = {

    # ================= FORMAÇÃO =================
    # Iniciação
    "IC": "Formação - Iniciação Científica",
    "ICJ": "Formação - Iniciação Científica",
    "ITI": "Formação - Iniciação Científica",
    "IT": "Formação - Iniciação Científica",
    "ITC": "Formação - Iniciação Científica",
    "IEX": "Formação - Iniciação Científica",

    # Mestrado
    "GM": "Formação - Mestrado",
    "MPE": "Formação - Mestrado",

    # Doutorado (Brasil)
    "GD": "Formação - Doutorado",

    # ================= MOBILIDADE =================
    # Doutorado sanduíche / exterior
    "SWE": "Mobilidade - Exterior",
    "SWI": "Mobilidade - Exterior",
    "SWP": "Mobilidade - Exterior",
    "GDE": "Mobilidade - Exterior",

    # Estágios / visitante / exterior geral
    "ESN": "Mobilidade - Exterior",
    "EJR": "Mobilidade - Exterior",
    "BSP": "Mobilidade - Exterior",
    "BEP": "Mobilidade - Exterior",
    "SPE": "Mobilidade - Exterior",
    "MSP": "Mobilidade - Exterior",
    "MEP": "Mobilidade - Exterior",

    # ================= CARREIRA CIENTÍFICA =================
    # Pós-doc
    "PD": "Carreira - Pós-Doutorado",
    "PDE": "Carreira - Pós-Doutorado",
    "PDJ": "Carreira - Pós-Doutorado",
    "PDS": "Carreira - Pós-Doutorado",
    "PDI": "Carreira - Pós-Doutorado",
    "PDT": "Carreira - Pós-Doutorado",
    "PDP": "Carreira - Pós-Doutorado",

    # Produtividade / pesquisador
    "PQ": "Carreira - Produtividade",
    "DT": "Carreira - Produtividade",

    # Fixação / pesquisador visitante
    "DCR": "Carreira - Fixação de Pesquisador",
    "RD": "Carreira - Fixação de Pesquisador",
    "PV": "Carreira - Fixação de Pesquisador",
    "PVE": "Carreira - Fixação de Pesquisador",

    # ================= PESQUISA =================
    # Projetos e financiamento direto
    "APQ": "Pesquisa - Auxílio à Pesquisa",
    "ARC": "Pesquisa - Auxílio à Pesquisa",
    "AED": "Pesquisa - Auxílio à Pesquisa",
    "ADC": "Pesquisa - Auxílio à Pesquisa",
    "AI": "Pesquisa - Auxílio à Pesquisa",

    # Apoio técnico (separado porque é diferente de projeto)
    "AT": "Pesquisa - Apoio Técnico",
    "ATP": "Pesquisa - Apoio Técnico",

    # ================= TECNOLOGIA & INOVAÇÃO =================
    "DTI": "Tecnologia - Desenvolvimento",
    "DES": "Tecnologia - Desenvolvimento",
    "DEJ": "Tecnologia - Desenvolvimento",
    "DTC": "Tecnologia - Desenvolvimento",
    "SDT": "Tecnologia - Desenvolvimento",
    "INT": "Tecnologia - Desenvolvimento",
    "PIN": "Tecnologia - Desenvolvimento",

    # ================= EVENTOS / DIFUSÃO =================
    "AVG": "Difusão - Eventos Científicos",
    "EXP": "Difusão - Extensão",
    "EV": "Difusão - Visitante",
    "BEV": "Difusão - Visitante",

    # ================= FALLBACK CONTROLADO =================
    # (casos que aparecem no dataset mas são raros)
    "SET": "Outros",
    "PCI": "Outros",
    "ACT": "Outros",
    "BJT": "Outros",
    "FIX": "Outros",
    "MAT": "Outros",
    "MDT": "Outros",
}

In [27]:
def normalizar(texto):
    if pd.isna(texto):
        return ""
    
    texto = str(texto).upper()
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('ASCII')
    texto = re.sub(r'[^A-Z\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto


def classificar_modalidade(mod):
    if pd.isna(mod):
        return "Não informado"

    mod_str = str(mod).upper()

    mod_norm = normalizar(mod_str)

    # ===== FORMAÇÃO =====
    if "POS DOUTORADO" in mod_norm:
        return "Carreira - Pós-Doutorado"

    elif "DOUTORADO" in mod_norm:
        # cuidado: pode ser sanduíche/exterior
        if "EXTERIOR" in mod_norm or "SANDUICHE" in mod_norm:
            return "Mobilidade - Doutorado - Exterior"
        return "Formação - Doutorado"

    elif "MESTRADO" in mod_norm:
        return "Formação - Mestrado"

    elif "INICIACAO" in mod_norm:
        return "Formação - Iniciação Científica"


    # ===== CARREIRA =====
    elif "PRODUTIVIDADE" in mod_norm:
        return "Carreira - Produtividade"


    # ===== TECNOLOGIA =====
    elif "DESENVOLVIMENTO" in mod_norm or "TECNOLOGICO" in mod_norm:
        return "Tecnologia - Desenvolvimento"


    # ===== PESQUISA =====
    elif "APOIO" in mod_norm or "AUXILIO" in mod_norm:
        return "Pesquisa - Auxílio à Pesquisa"


    # ===== MOBILIDADE (fallback textual) =====
    elif "EXTERIOR" in mod_norm or "VISITANTE" in mod_norm:
        return "Mobilidade - Exterior"


    return "Outros"





In [28]:
df['categoria'] = df['modalidade'].apply(classificar_modalidade)
df['ano_referencia'] = pd.to_numeric(df['ano_referencia'], errors='coerce') 
df['valor_pago'] = pd.to_numeric(df['valor_pago'], errors='coerce')


In [29]:
print("\nCategorias criadas:")
print(df['categoria'].value_counts())
df.head()




Categorias criadas:
categoria
Formação - Iniciação Científica      2554
Carreira - Produtividade              626
Formação - Mestrado                   429
Formação - Doutorado                  403
Pesquisa - Auxílio à Pesquisa         298
Outros                                156
Tecnologia - Desenvolvimento          155
Carreira - Pós-Doutorado               71
Mobilidade - Exterior                  48
Mobilidade - Doutorado - Exterior      10
Name: count, dtype: int64


,ano_referencia,linha_de_fomento,modalidade,programa_cnpq,grande_area,area,subarea,instituicao_origem,sigla_uf_origem,pais_origem,...,sigla_instituicao_destino,sigla_instituicao_macro,cidade_destino,sigla_uf_destino,regiao_destino,pais_destino,titulo_do_projeto,palavra_chave,valor_pago,categoria
2032718,2012.0,Bolsas de Produtividade em Pesquisa e Tecnologia,PQ - Produtividade em Pesquisa,PROGRAMA BASICO DE PARASITOLOGIA,Ciências Biológicas,Parasitologia,Protozoologia de Parasitos,Universidade de São Paulo,SP,BRA - Brasil,...,USP,USP,São Paulo,SP,NaN,BRA - Brasil,NaN,NaN,24000.0,Carreira - Produtividade
2948491,2025.0,0.0 - Problemas Encontrar Modalidade Processo,PQ -,Programa Regular de Bolsas de Produtividade em...,Ciências da Saúde,Enfermagem,Enfermagem Médico-Cirúrgica,Escola de Enfermagem de Ribeirao Preto,SP,BRA - Brasil,...,USP/EERP,USP,Ribeirao Preto,SP,SE,BRA - Brasil,NaN,NaN,7860.0,Outros
131056,2024.0,BOLSAS DE FORMAÇÃO E DE PESQUISADORES,PQ - Produtividade em Pesquisa,Programa Regular de Bolsas de Produtividade em...,Ciências Exatas e da Terra,Física,Física das Partículas Elementares e Campos,Centro Brasileiro de Pesquisas Fisicas,RJ,BRA - Brasil,...,CBPF,CBPF,Rio de Janeiro,RJ,SE,BRA - Brasil,Instrumentação e Análise de dados em Física de...,Violação de CP;detectores;eletrônica;mudanças ...,4200.0,Carreira - Produtividade
894537,2018.0,Bolsas de Produtividade em Pesquisa e Tecnologia,PQ - Produtividade em Pesquisa,PROGRAMA BASICO DE AGRONOMIA,Ciências Agrárias,Agronomia,Fitotecnia,Universidade Federal do Rio Grande do Sul,RS,BRA - Brasil,...,UFRGS,UFRGS,Porto Alegre,RS,NaN,BRA - Brasil,NaN,NaN,26400.0,Carreira - Produtividade
2535404,2009.0,Bolsas de Doutorado,GD - Doutorado,PROGRAMA DE POS GRADUAÇÃO,Ciências Biológicas,Biologia Geral,Biologia Geral,NaN,NaN,-,...,USP,USP,São Paulo,SP,NaN,BRA - Brasil,NaN,NaN,26328.0,Formação - Doutorado


In [30]:


df[['p1', 'p2', 'p3',]] = df['palavra_chave'].str.split(';', n=2, expand=True)
df[['p1', 'p2', 'p3']] = df[['p1', 'p2', 'p3']].fillna('')

for col in ['p1', 'p2', 'p3']:
    df[col] = df[col].str.strip().str.upper()



print("\nExemplo de palavras-chave extraídas:")
print(df[['palavra_chave', 'p1', 'p2', 'p3']].head())
df.drop(columns=['palavra_chave', 'modalidade'], inplace=True)

df.head()


Exemplo de palavras-chave extraídas:
                                             palavra_chave              p1  \
2032718                                                NaN                   
2948491                                                NaN                   
131056   Violação de CP;detectores;eletrônica;mudanças ...  VIOLAÇÃO DE CP   
894537                                                 NaN                   
2535404                                                NaN                   

                 p2                                                 p3  
2032718                                                                 
2948491                                                                 
131056   DETECTORES  ELETRÔNICA;MUDANÇAS CLIMÁTICAS;VIDA-MÉDIA DE R...  
894537                                                                  
2535404                                                                 


,ano_referencia,linha_de_fomento,programa_cnpq,grande_area,area,subarea,instituicao_origem,sigla_uf_origem,pais_origem,instituicao_destino,...,cidade_destino,sigla_uf_destino,regiao_destino,pais_destino,titulo_do_projeto,valor_pago,categoria,p1,p2,p3
2032718,2012.0,Bolsas de Produtividade em Pesquisa e Tecnologia,PROGRAMA BASICO DE PARASITOLOGIA,Ciências Biológicas,Parasitologia,Protozoologia de Parasitos,Universidade de São Paulo,SP,BRA - Brasil,Universidade de São Paulo,...,São Paulo,SP,NaN,BRA - Brasil,NaN,24000.0,Carreira - Produtividade,,,
2948491,2025.0,0.0 - Problemas Encontrar Modalidade Processo,Programa Regular de Bolsas de Produtividade em...,Ciências da Saúde,Enfermagem,Enfermagem Médico-Cirúrgica,Escola de Enfermagem de Ribeirao Preto,SP,BRA - Brasil,Escola de Enfermagem de Ribeirao Preto,...,Ribeirao Preto,SP,SE,BRA - Brasil,NaN,7860.0,Outros,,,
131056,2024.0,BOLSAS DE FORMAÇÃO E DE PESQUISADORES,Programa Regular de Bolsas de Produtividade em...,Ciências Exatas e da Terra,Física,Física das Partículas Elementares e Campos,Centro Brasileiro de Pesquisas Fisicas,RJ,BRA - Brasil,Centro Brasileiro de Pesquisas Fisicas,...,Rio de Janeiro,RJ,SE,BRA - Brasil,Instrumentação e Análise de dados em Física de...,4200.0,Carreira - Produtividade,VIOLAÇÃO DE CP,DETECTORES,ELETRÔNICA;MUDANÇAS CLIMÁTICAS;VIDA-MÉDIA DE R...
894537,2018.0,Bolsas de Produtividade em Pesquisa e Tecnologia,PROGRAMA BASICO DE AGRONOMIA,Ciências Agrárias,Agronomia,Fitotecnia,Universidade Federal do Rio Grande do Sul,RS,BRA - Brasil,Universidade Federal do Rio Grande do Sul,...,Porto Alegre,RS,NaN,BRA - Brasil,NaN,26400.0,Carreira - Produtividade,,,
2535404,2009.0,Bolsas de Doutorado,PROGRAMA DE POS GRADUAÇÃO,Ciências Biológicas,Biologia Geral,Biologia Geral,NaN,NaN,-,Universidade de São Paulo,...,São Paulo,SP,NaN,BRA - Brasil,NaN,26328.0,Formação - Doutorado,,,


In [31]:
# =====================================================
# SPLIT
# =====================================================

X = df.drop(columns=[TARGET])

y = pd.to_numeric(
    df[TARGET],
    errors="coerce"
)

valid = y.notna()

X = X.loc[valid]
y = y.loc[valid]

# Aplicar transformação logarítmica para reduzir impacto de outliers
y = np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("\nTreino:", X_train.shape)
print("Teste :", X_test.shape)



Treino: (3800, 21)
Teste : (950, 21)


In [32]:
print(X_train.columns[2]) 

programa_cnpq


In [ ]:
import numpy as np
import pandas as pd
import torch  # Substituímos tensorflow por torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# 1. Separe as colunas textuais de verdade
text_cols = [
    'categoria', 'p1', 'p2', 'p3', 'titulo_do_projeto',
    'sigla_uf_destino', 'sigla_instituicao_destino',
    'grande_area', 'area', 'subarea', 'linha_de_fomento',
    'programa_cnpq', 'instituicao_origem', 'cidade_destino', 'sigla_instituicao_macro',
    'cidade_destino',
    'sigla_uf_origem',
    'regiao_destino',
    'pais_destino',
    'pais_origem',
    'instituicao_destino',
    


    
]

num_cols = [c for c in X.columns if c not in text_cols]

# 2. Crie um texto único por linha
X_text = (
    X[text_cols]
    .fillna('')
    .astype(str)
    .agg(' '.join, axis=1)
)

X_num = X[num_cols].copy()
X_num = X_num.fillna(0)

# 3. Carregue BERT
model_name = 'neuralmind/bert-base-portuguese-cased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)

# Colocamos o modelo em modo de avaliação (boa prática)
bert_model.eval()

# 4. Função para gerar embeddings
def get_bert_embeddings(texts, max_length=128, batch_size=16):
    embeddings = []
    texts = list(texts)

    # USAR NO_GRAD É ESSENCIAL PARA NÃO ESTOURAR A MEMÓRIA RAM
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            tokens = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'  # <-- MUDAMOS DE 'tf' PARA 'pt'
            )

            # Passamos os tokens desempacotados (**tokens) para o modelo PyTorch
            outputs = bert_model(**tokens)
            
            # Pegamos o token [CLS] (índice 0)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]
            
            # Convertendo de volta para numpy
            embeddings.append(cls_embeddings.cpu().numpy())
            
    print(f"Processados {len(texts)} / {len(X_text)} textos.")
    return np.vstack(embeddings)

# 5. Gere os embeddings
print("\nGerando embeddings BERT (pode levar um tempo)...")
X_bert = get_bert_embeddings(X_text)

# 6. Junte embeddings + features numéricas
X_final = np.hstack([X_num.values, X_bert])
print("Shape final dos dados para o modelo:", X_final.shape)

# 7. Treine o modelo final
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42
)

# model = CatBoostRegressor(verbose=100)
# model.fit(X_train, y_train)

# print("\nModelo treinado. Fazendo previsões e avaliando...")
# preds = model.predict(X_test)

# print("\nResultados:")
# print("MAE:", mean_absolute_error(y_test, preds))
# print("RMSE:", root_mean_squared_error(y_test, preds))
# print("R²:", r2_score(y_test, preds))

# agora random forest com os mesmos dados para comparação
print("\nTreinando Random Forest para comparação...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
print("\nResultados do Random Forest:")
print("MAE:", mean_absolute_error(y_test, rf_preds))
print("RMSE:", root_mean_squared_error(y_test, rf_preds))
print("R²:", r2_score(y_test, rf_preds))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Gerando embeddings BERT (pode levar um tempo)...
Processados 4750 / 4750 textos.
Shape final dos dados para o modelo: (4750, 769)
Learning rate set to 0.050557
0:	learn: 1.3175530	total: 156ms	remaining: 2m 35s
100:	learn: 0.7985169	total: 5.94s	remaining: 52.8s
200:	learn: 0.6852034	total: 11.7s	remaining: 46.6s
300:	learn: 0.5855502	total: 17.5s	remaining: 40.6s
400:	learn: 0.5144107	total: 23s	remaining: 34.3s
500:	learn: 0.4578653	total: 28.4s	remaining: 28.2s
600:	learn: 0.4106105	total: 33.7s	remaining: 22.4s
700:	learn: 0.3729559	total: 39.1s	remaining: 16.7s
800:	learn: 0.3392628	total: 44.5s	remaining: 11s
900:	learn: 0.3126161	total: 49.9s	remaining: 5.49s
999:	learn: 0.2905460	total: 55.6s	remaining: 0us

Modelo treinado. Fazendo previsões e avaliando...

Resultados:
MAE: 0.6276372259927864
RMSE: 0.8175149757528146
R²: 0.5957669247603365
